# NB11：可觀測性與追蹤 — 讓 RAG 與 Agent 系統透明化

**目標：** 為 RAG 與 Agent pipeline 建立完整的可觀測性基礎設施，包含結構化日誌、LLM-native 指標、分散式追蹤，以及告警與預算控制機制。

**學習成果：**
- 為 RAG/Agent pipeline 每個階段實作結構化日誌
- 追蹤延遲細項：檢索 / 重排序 / LLM 生成 / 總計
- 追蹤 LLM-native 指標：tokens/sec、cost-per-request、每日成本推算
- 從零實作 OpenTelemetry 風格的 Span Tracing（不依賴 OTel 套件）
- 建立文字 + matplotlib 儀表板摘要
- 實作告警邏輯：延遲超過閾值或成本超過預算時觸發警示

---
**使用技術：** OpenAI gpt-4o-mini、text-embedding-3-small、ChromaDB（in-memory）、純 Python（無 LangChain）

## Part 0：環境設定

In [ ]:
# 安裝必要套件（如尚未安裝）
# !pip install openai chromadb python-dotenv matplotlib

In [ ]:
import os
import json
import uuid
import time
import random
import logging
import datetime
import statistics
from typing import Optional, Dict, List, Any
from dataclasses import dataclass, field, asdict
from collections import defaultdict

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from dotenv import load_dotenv
import openai
import chromadb

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

client = openai.OpenAI(api_key=OPENAI_API_KEY)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL  = "gpt-4o-mini"

# OpenAI gpt-4o-mini 定價 (USD per 1K tokens, 截至 2025)
PRICE_INPUT_PER_1K  = 0.000150   # $0.15 / 1M tokens
PRICE_OUTPUT_PER_1K = 0.000600   # $0.60 / 1M tokens

print("✅ 環境設定完成")
print(f"  Chat model  : {CHAT_MODEL}")
print(f"  Embed model : {EMBED_MODEL}")

---
## Part 1：為什麼可觀測性很重要

### 1.1 沒有可觀測性，你看不見什麼？

在生產環境中，RAG 系統常見的「隱形問題」：

| 問題類型 | 沒有可觀測性的症狀 | 有可觀測性後能看到 |
|---------|------------------|------------------|
| 檢索品質下降 | 用戶抱怨答案不準確 | Retrieval recall@k 下滑、rerank 分數分布偏移 |
| 延遲爆炸 | 頁面轉圈很久 | 哪個 span 耗時超標（Embedding vs LLM） |
| 成本失控 | 月底帳單超標 | 每日 token 消耗趨勢、高成本請求識別 |
| Hallucination 增加 | 用戶回報錯誤資訊 | Context 長度 vs faithfulness 分數相關性 |
| 快取未命中 | 重複查詢仍呼叫 LLM | Cache hit rate 低，semantic cache 未發揮效用 |

### 1.2 可觀測性三支柱

```
┌─────────────────────────────────────────────────────────────┐
│                   Observability Pillars                      │
│                                                             │
│  📋 Logs          📊 Metrics         🔍 Traces              │
│  ─────────        ─────────          ────────               │
│  離散事件記錄      時序數值聚合        請求的因果鏈            │
│  「發生了什麼」    「趨勢如何」        「為什麼這麼慢」         │
│                                                             │
│  structured JSON  counters/histograms  spans + trace_id     │
└─────────────────────────────────────────────────────────────┘
```

### 1.3 LLM-Specific 指標

傳統 Web 服務的 SLI（如 RPS、Error Rate、Latency）對 LLM 系統不夠用，還需要：

- **tokens/sec**（throughput）：LLM 生成速度，影響用戶感知
- **cost-per-request**：每次查詢的美元成本，驅動架構決策
- **cache hit rate**：Semantic cache 或 Prompt cache 是否發揮作用
- **hallucination rate**：需結合評估框架（NB04）
- **context utilization**：送進去的 token 有多少是有效的

---
## Part 2：結構化日誌（Structured Logging）

### 2.1 為什麼要用結構化日誌？

傳統 `print("Retrieval done in 120ms")` 的問題：
- 無法用程式查詢（grep 只能搜字串）
- 沒有 request_id，無法關聯同一請求的多個步驟
- 無法匯入 Elasticsearch、BigQuery 做分析

**結構化日誌** = 每條 log 是一個 JSON 物件，欄位固定可查詢。

In [ ]:
# ─────────────────────────────────────────────
# PipelineLogger：結構化日誌類別
# ─────────────────────────────────────────────

@dataclass
class LogEntry:
    """單筆結構化日誌"""
    timestamp: str
    request_id: str
    step_name: str
    duration_ms: float
    tokens_in: int = 0
    tokens_out: int = 0
    status: str = "ok"          # ok | error | skipped
    error: Optional[str] = None
    extra: Dict[str, Any] = field(default_factory=dict)

    def to_json(self) -> str:
        return json.dumps(asdict(self), ensure_ascii=False)


class PipelineLogger:
    """為 RAG pipeline 每個步驟記錄結構化日誌"""

    def __init__(self, sink: Optional[List] = None):
        """
        sink: 若提供 list，日誌物件也會 append 進去（方便測試）
        """
        self._entries: List[LogEntry] = []
        self._sink = sink  # 外部容器

    def log(
        self,
        request_id: str,
        step_name: str,
        duration_ms: float,
        tokens_in: int = 0,
        tokens_out: int = 0,
        status: str = "ok",
        error: Optional[str] = None,
        **extra,
    ) -> LogEntry:
        entry = LogEntry(
            timestamp=datetime.datetime.utcnow().isoformat() + "Z",
            request_id=request_id,
            step_name=step_name,
            duration_ms=round(duration_ms, 2),
            tokens_in=tokens_in,
            tokens_out=tokens_out,
            status=status,
            error=error,
            extra=extra,
        )
        self._entries.append(entry)
        if self._sink is not None:
            self._sink.append(entry)
        print(entry.to_json())
        return entry

    def query(
        self,
        step_name: Optional[str] = None,
        status: Optional[str] = None,
        request_id: Optional[str] = None,
    ) -> List[LogEntry]:
        """按欄位過濾日誌（模擬 log 查詢）"""
        results = self._entries
        if step_name:
            results = [e for e in results if e.step_name == step_name]
        if status:
            results = [e for e in results if e.status == status]
        if request_id:
            results = [e for e in results if e.request_id == request_id]
        return results

    def all_entries(self) -> List[LogEntry]:
        return list(self._entries)


logger = PipelineLogger()
print("✅ PipelineLogger 初始化完成")

### 2.2 建立 ChromaDB 知識庫

In [ ]:
# ─────────────────────────────────────────────
# ChromaDB in-memory 知識庫
# ─────────────────────────────────────────────

DOCS = [
    "OpenTelemetry 是一個開源的可觀測性框架，提供統一的 API、SDK 和工具，用於收集 Traces、Metrics 和 Logs。",
    "分散式追蹤（Distributed Tracing）透過 trace_id 將跨服務的請求關聯起來，幫助開發者找到效能瓶頸。",
    "Span 是追蹤的基本單位，記錄一個操作的開始時間、結束時間、屬性和狀態。",
    "結構化日誌使用 JSON 格式記錄事件，每個欄位都有固定的型別，可以用程式查詢和分析。",
    "P95 延遲代表 95% 的請求都在這個時間內完成，是比平均值更能反映用戶體驗的指標。",
    "tokens/sec 是衡量 LLM 生成速度的指標，受到模型大小、硬體規格和批次大小影響。",
    "Cost-per-request 計算方式：(input_tokens * 輸入單價 + output_tokens * 輸出單價) / 1000。",
    "告警系統應該設定明確的閾值，避免告警疲勞，建議區分 WARNING 和 CRITICAL 嚴重程度。",
    "Circuit Breaker（斷路器）模式可以防止系統在異常時持續耗費資源，當錯誤率超過閾值時自動停止請求。",
    "RAG 系統的延遲通常由三部分組成：向量搜尋（< 100ms）、LLM 生成（500ms - 5s）、後處理（< 50ms）。",
]

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="observability_kb",
    metadata={"hnsw:space": "cosine"},
)

def embed(texts: List[str]) -> List[List[float]]:
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in resp.data]

embeddings = embed(DOCS)
collection.add(
    ids=[f"doc_{i}" for i in range(len(DOCS))],
    documents=DOCS,
    embeddings=embeddings,
)

print(f"✅ 已載入 {len(DOCS)} 筆文件到 ChromaDB")

### 2.3 帶有結構化日誌的 RAG Pipeline

In [ ]:
def rag_with_logging(
    query: str,
    pipeline_logger: PipelineLogger,
    top_k: int = 3,
) -> Dict[str, Any]:
    """帶有結構化日誌的 RAG pipeline"""

    request_id = str(uuid.uuid4())[:8]
    result = {"request_id": request_id, "query": query}

    # ── Step 1：Embedding 查詢 ──────────────────────
    t0 = time.perf_counter()
    try:
        q_emb = embed([query])[0]
        embed_ms = (time.perf_counter() - t0) * 1000
        pipeline_logger.log(
            request_id=request_id,
            step_name="embedding",
            duration_ms=embed_ms,
            model=EMBED_MODEL,
        )
    except Exception as e:
        pipeline_logger.log(request_id, "embedding", 0, status="error", error=str(e))
        result["error"] = str(e)
        return result

    # ── Step 2：向量檢索 ───────────────────────────
    t0 = time.perf_counter()
    chroma_res = collection.query(query_embeddings=[q_emb], n_results=top_k)
    retrieve_ms = (time.perf_counter() - t0) * 1000
    docs = chroma_res["documents"][0]
    pipeline_logger.log(
        request_id=request_id,
        step_name="retrieval",
        duration_ms=retrieve_ms,
        top_k=top_k,
        docs_returned=len(docs),
    )

    # ── Step 3：LLM 生成 ───────────────────────────
    context = "\n".join(f"- {d}" for d in docs)
    messages = [
        {"role": "system", "content": "你是一個可觀測性專家，請根據提供的知識庫內容回答問題，回答請使用繁體中文。"},
        {"role": "user",   "content": f"知識庫：\n{context}\n\n問題：{query}"},
    ]
    t0 = time.perf_counter()
    try:
        resp = client.chat.completions.create(
            model=CHAT_MODEL, messages=messages, max_tokens=300, temperature=0.3
        )
        llm_ms = (time.perf_counter() - t0) * 1000
        usage = resp.usage
        answer = resp.choices[0].message.content

        pipeline_logger.log(
            request_id=request_id,
            step_name="llm_generation",
            duration_ms=llm_ms,
            tokens_in=usage.prompt_tokens,
            tokens_out=usage.completion_tokens,
            model=CHAT_MODEL,
        )
        result["answer"] = answer
        result["usage"] = {
            "tokens_in": usage.prompt_tokens,
            "tokens_out": usage.completion_tokens,
            "llm_ms": llm_ms,
        }
    except Exception as e:
        pipeline_logger.log(request_id, "llm_generation", 0, status="error", error=str(e))
        result["error"] = str(e)

    return result


print("✅ rag_with_logging() 定義完成")

In [ ]:
# ─── 執行示範 ────────────────────────────────
demo_logger = PipelineLogger()
result = rag_with_logging("什麼是 P95 延遲？", demo_logger)

print("\n📌 答案：")
print(result.get("answer", "（無答案）"))

In [ ]:
# ─── 查詢日誌：過濾出 llm_generation 步驟 ───
print("📋 查詢 step_name='llm_generation' 的日誌：")
for entry in demo_logger.query(step_name="llm_generation"):
    print(f"  request_id={entry.request_id}  "
          f"duration={entry.duration_ms:.1f}ms  "
          f"tokens_in={entry.tokens_in}  tokens_out={entry.tokens_out}")

---
## Part 3：LLM-Native 指標追蹤

### 3.1 MetricsCollector 設計

每次請求後記錄：
- `tokens_in` / `tokens_out`
- `cost_usd` = `(tokens_in * 0.00015 + tokens_out * 0.0006) / 1000`
- `latency_ms`（LLM 生成延遲）
- `tokens_per_sec` = `tokens_out / (latency_ms / 1000)`

In [ ]:
@dataclass
class RequestMetric:
    request_id: str
    timestamp: float           # Unix timestamp
    tokens_in: int
    tokens_out: int
    cost_usd: float
    latency_ms: float          # 總端到端延遲
    llm_latency_ms: float      # 只計 LLM 部分
    tokens_per_sec: float


class MetricsCollector:
    """累積並分析 LLM-native 指標"""

    def __init__(self):
        self._metrics: List[RequestMetric] = []

    def record(
        self,
        request_id: str,
        tokens_in: int,
        tokens_out: int,
        latency_ms: float,
        llm_latency_ms: float,
    ) -> RequestMetric:
        cost_usd = (tokens_in * PRICE_INPUT_PER_1K + tokens_out * PRICE_OUTPUT_PER_1K) / 1000
        tps = tokens_out / (llm_latency_ms / 1000) if llm_latency_ms > 0 else 0.0
        m = RequestMetric(
            request_id=request_id,
            timestamp=time.time(),
            tokens_in=tokens_in,
            tokens_out=tokens_out,
            cost_usd=cost_usd,
            latency_ms=latency_ms,
            llm_latency_ms=llm_latency_ms,
            tokens_per_sec=tps,
        )
        self._metrics.append(m)
        return m

    # ── 統計方法 ──────────────────────────────────

    def _latencies(self) -> List[float]:
        return [m.latency_ms for m in self._metrics]

    def percentile(self, p: float) -> float:
        lats = sorted(self._latencies())
        if not lats:
            return 0.0
        idx = int(len(lats) * p / 100)
        idx = min(idx, len(lats) - 1)
        return lats[idx]

    def summary(self) -> Dict[str, Any]:
        if not self._metrics:
            return {}
        lats = self._latencies()
        costs = [m.cost_usd for m in self._metrics]
        tps_vals = [m.tokens_per_sec for m in self._metrics]
        total_cost = sum(costs)
        # 推算：假設每天 1000 個請求
        avg_cost = statistics.mean(costs)
        projected_daily  = avg_cost * 1000
        projected_monthly = projected_daily * 30

        return {
            "requests": len(self._metrics),
            "avg_latency_ms": round(statistics.mean(lats), 1),
            "p50_latency_ms": round(self.percentile(50), 1),
            "p95_latency_ms": round(self.percentile(95), 1),
            "avg_tokens_in": round(statistics.mean([m.tokens_in for m in self._metrics]), 1),
            "avg_tokens_out": round(statistics.mean([m.tokens_out for m in self._metrics]), 1),
            "avg_tokens_per_sec": round(statistics.mean(tps_vals), 1),
            "avg_cost_usd": round(avg_cost, 6),
            "total_cost_usd": round(total_cost, 6),
            "projected_daily_usd": round(projected_daily, 4),
            "projected_monthly_usd": round(projected_monthly, 2),
        }

    def all_metrics(self) -> List[RequestMetric]:
        return list(self._metrics)


metrics = MetricsCollector()
print("✅ MetricsCollector 初始化完成")

In [ ]:
# ─── 帶有指標收集的 RAG ──────────────────────

def rag_with_metrics(
    query: str,
    pipeline_logger: PipelineLogger,
    collector: MetricsCollector,
    top_k: int = 3,
) -> Dict[str, Any]:
    """帶有日誌 + 指標的完整 RAG"""
    t_total_start = time.perf_counter()
    request_id = str(uuid.uuid4())[:8]

    # Embedding
    t0 = time.perf_counter()
    q_emb = embed([query])[0]
    embed_ms = (time.perf_counter() - t0) * 1000
    pipeline_logger.log(request_id, "embedding", embed_ms)

    # Retrieval
    t0 = time.perf_counter()
    chroma_res = collection.query(query_embeddings=[q_emb], n_results=top_k)
    retrieve_ms = (time.perf_counter() - t0) * 1000
    docs = chroma_res["documents"][0]
    pipeline_logger.log(request_id, "retrieval", retrieve_ms, docs_returned=len(docs))

    # LLM
    context = "\n".join(f"- {d}" for d in docs)
    messages = [
        {"role": "system", "content": "你是可觀測性專家，請用繁體中文簡短回答。"},
        {"role": "user",   "content": f"知識庫：\n{context}\n\n問題：{query}"},
    ]
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=CHAT_MODEL, messages=messages, max_tokens=200, temperature=0.3
    )
    llm_ms = (time.perf_counter() - t0) * 1000
    usage = resp.usage
    pipeline_logger.log(
        request_id, "llm_generation", llm_ms,
        tokens_in=usage.prompt_tokens, tokens_out=usage.completion_tokens
    )

    total_ms = (time.perf_counter() - t_total_start) * 1000

    # 記錄到 MetricsCollector
    m = collector.record(
        request_id=request_id,
        tokens_in=usage.prompt_tokens,
        tokens_out=usage.completion_tokens,
        latency_ms=total_ms,
        llm_latency_ms=llm_ms,
    )

    return {
        "request_id": request_id,
        "answer": resp.choices[0].message.content,
        "metric": m,
    }


print("✅ rag_with_metrics() 定義完成")

In [ ]:
# ─── 跑幾筆請求，收集指標 ──────────────────────
test_queries = [
    "什麼是結構化日誌？",
    "如何計算 cost-per-request？",
    "Span 是什麼？",
]

metrics_logger = PipelineLogger()
for q in test_queries:
    r = rag_with_metrics(q, metrics_logger, metrics)
    m = r["metric"]
    print(f"\n✅ [{r['request_id']}] {q[:30]}")
    print(f"   cost=${m.cost_usd:.6f}  latency={m.latency_ms:.0f}ms  "
          f"tokens/sec={m.tokens_per_sec:.1f}")

In [ ]:
# ─── 顯示彙總統計 ─────────────────────────────
s = metrics.summary()
print("\n📊 MetricsCollector 統計摘要")
print("─" * 45)
for k, v in s.items():
    print(f"  {k:<30} {v}")

### 3.2 視覺化：延遲分布 & 成本時序

In [ ]:
def plot_metrics(collector: MetricsCollector):
    """繪製延遲直方圖與成本時序圖"""
    ms_list = collector.all_metrics()
    if len(ms_list) < 2:
        print("⚠️ 需要至少 2 筆資料才能繪圖")
        return

    lats = [m.latency_ms for m in ms_list]
    costs = [m.cost_usd for m in ms_list]
    cum_costs = [sum(costs[: i + 1]) for i in range(len(costs))]
    indices = list(range(1, len(ms_list) + 1))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("LLM-Native 指標儀表板", fontsize=14, fontweight="bold")

    # ── 左：延遲直方圖 ──────────────────────────
    axes[0].hist(lats, bins=max(3, len(lats) // 2), color="steelblue", edgecolor="white", alpha=0.85)
    p50 = statistics.median(lats)
    p95 = sorted(lats)[min(int(len(lats) * 0.95), len(lats) - 1)]
    axes[0].axvline(p50, color="orange", linestyle="--", label=f"P50={p50:.0f}ms")
    axes[0].axvline(p95, color="red",    linestyle="--", label=f"P95={p95:.0f}ms")
    axes[0].set_xlabel("端到端延遲 (ms)")
    axes[0].set_ylabel("請求數")
    axes[0].set_title("延遲分布直方圖")
    axes[0].legend()

    # ── 右：累積成本時序 ────────────────────────
    axes[1].plot(indices, cum_costs, marker="o", color="coral", linewidth=2)
    axes[1].fill_between(indices, cum_costs, alpha=0.2, color="coral")
    axes[1].set_xlabel("請求序號")
    axes[1].set_ylabel("累積成本 (USD)")
    axes[1].set_title("累積成本時序")
    axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.5f"))

    plt.tight_layout()
    plt.show()


plot_metrics(metrics)

---
## Part 4：分散式追蹤（Trace & Span）

### 4.1 OpenTelemetry 概念回顧

```
Trace（追蹤）= 一個完整請求的所有 Span
  └─ Span（跨度）= 一個操作的時間區間
       ├─ trace_id      ← 整個 trace 共用
       ├─ span_id       ← 每個 span 唯一
       ├─ parent_span_id ← 構成樹狀關係
       ├─ name          ← 操作名稱
       ├─ start_time / end_time
       ├─ attributes    ← 鍵值對 metadata
       └─ status        ← ok / error
```

**為什麼需要 Span Tree？**
- 可以看到 retrieval span 占了多少比例
- 可以看到 LLM span 的 token 用量
- parent_span_id 可以重建因果關係

In [ ]:
# ─────────────────────────────────────────────
# 從零實作 Tracer / Span（不依賴 OTel 套件）
# ─────────────────────────────────────────────

@dataclass
class Span:
    trace_id: str
    span_id: str
    parent_span_id: Optional[str]
    name: str
    start_time: float            # Unix timestamp (ns 精度)
    end_time: Optional[float] = None
    attributes: Dict[str, Any] = field(default_factory=dict)
    status: str = "ok"           # ok | error
    error_msg: Optional[str] = None

    @property
    def duration_ms(self) -> float:
        if self.end_time is None:
            return 0.0
        return (self.end_time - self.start_time) * 1000

    def end(self, status: str = "ok", error_msg: Optional[str] = None, **attrs):
        """結束此 Span，可附加額外 attributes"""
        self.end_time = time.perf_counter()
        self.status = status
        if error_msg:
            self.error_msg = error_msg
        self.attributes.update(attrs)

    def to_dict(self) -> Dict:
        return {
            "trace_id": self.trace_id,
            "span_id": self.span_id,
            "parent_span_id": self.parent_span_id,
            "name": self.name,
            "duration_ms": round(self.duration_ms, 2),
            "status": self.status,
            "error_msg": self.error_msg,
            "attributes": self.attributes,
        }


class Tracer:
    """輕量級 Tracer，管理 Span 的建立與匯出"""

    def __init__(self):
        self._spans: List[Span] = []

    def start_span(
        self,
        name: str,
        trace_id: Optional[str] = None,
        parent_span: Optional[Span] = None,
        **attrs,
    ) -> Span:
        """建立並回傳新 Span（尚未結束）"""
        if trace_id is None:
            trace_id = uuid.uuid4().hex[:16]

        span = Span(
            trace_id=trace_id,
            span_id=uuid.uuid4().hex[:8],
            parent_span_id=parent_span.span_id if parent_span else None,
            name=name,
            start_time=time.perf_counter(),
            attributes=dict(attrs),
        )
        self._spans.append(span)
        return span

    def export(self, trace_id: Optional[str] = None) -> List[Dict]:
        """匯出指定 trace_id 的所有 span（或全部）"""
        spans = self._spans
        if trace_id:
            spans = [s for s in spans if s.trace_id == trace_id]
        return [s.to_dict() for s in spans]

    def print_trace_tree(self, trace_id: str):
        """以縮排方式印出 Span 樹"""
        spans = [s for s in self._spans if s.trace_id == trace_id]
        by_parent: Dict[Optional[str], List[Span]] = defaultdict(list)
        for s in spans:
            by_parent[s.parent_span_id].append(s)

        def _print(span_id: Optional[str], depth: int = 0):
            for span in by_parent.get(span_id, []):
                icon = "✅" if span.status == "ok" else "❌"
                indent = "  " * depth
                print(
                    f"{indent}{icon} [{span.span_id}] {span.name}"
                    f"  ({span.duration_ms:.1f}ms)"
                    + (f"  attrs={span.attributes}" if span.attributes else "")
                )
                _print(span.span_id, depth + 1)

        print(f"\n🔍 Trace: {trace_id}")
        print("─" * 60)
        _print(None)

    def all_spans(self) -> List[Span]:
        return list(self._spans)


tracer = Tracer()
print("✅ Tracer / Span 實作完成")

### 4.2 以 Span Tree 追蹤 RAG Pipeline

In [ ]:
def rag_with_tracing(
    query: str,
    tracer: Tracer,
    collector: MetricsCollector,
    top_k: int = 3,
) -> Dict[str, Any]:
    """帶有完整 Span 追蹤的 RAG Pipeline"""

    trace_id = uuid.uuid4().hex[:16]

    # ── Root Span ─────────────────────────────
    root = tracer.start_span("rag_pipeline", trace_id=trace_id, query=query[:50])

    # ── Embedding Span ────────────────────────
    emb_span = tracer.start_span("embedding", trace_id=trace_id, parent_span=root)
    q_emb = embed([query])[0]
    emb_span.end(model=EMBED_MODEL)

    # ── Retrieval Span ────────────────────────
    ret_span = tracer.start_span("retrieval", trace_id=trace_id, parent_span=root)
    chroma_res = collection.query(query_embeddings=[q_emb], n_results=top_k)
    docs = chroma_res["documents"][0]
    ret_span.end(top_k=top_k, returned=len(docs))

    # ── Rerank Span（模擬）────────────────────
    rerank_span = tracer.start_span("rerank", trace_id=trace_id, parent_span=root)
    time.sleep(0.01)   # 模擬 rerank 耗時
    rerank_span.end(strategy="mock_bm25", docs_after=len(docs))

    # ── LLM Generation Span ───────────────────
    context = "\n".join(f"- {d}" for d in docs)
    messages = [
        {"role": "system", "content": "你是可觀測性專家，請用繁體中文簡短回答。"},
        {"role": "user",   "content": f"知識庫：\n{context}\n\n問題：{query}"},
    ]
    llm_span = tracer.start_span("llm_generation", trace_id=trace_id, parent_span=root)
    resp = client.chat.completions.create(
        model=CHAT_MODEL, messages=messages, max_tokens=200, temperature=0.3
    )
    usage = resp.usage
    llm_span.end(
        model=CHAT_MODEL,
        tokens_in=usage.prompt_tokens,
        tokens_out=usage.completion_tokens,
    )

    # ── Root 結束 ─────────────────────────────
    root.end()

    # 記錄指標
    collector.record(
        request_id=trace_id[:8],
        tokens_in=usage.prompt_tokens,
        tokens_out=usage.completion_tokens,
        latency_ms=root.duration_ms,
        llm_latency_ms=llm_span.duration_ms,
    )

    return {
        "trace_id": trace_id,
        "answer": resp.choices[0].message.content,
    }


print("✅ rag_with_tracing() 定義完成")

In [ ]:
# ─── 執行並印出 Trace Tree ───────────────────
trace_metrics = MetricsCollector()
trace_result = rag_with_tracing(
    "P95 延遲代表什麼意思？",
    tracer,
    trace_metrics,
)

tracer.print_trace_tree(trace_result["trace_id"])

print("\n📌 答案：")
print(trace_result["answer"])

In [ ]:
# ─── 匯出 Trace 為 JSON ──────────────────────
exported = tracer.export(trace_id=trace_result["trace_id"])
print("📤 Trace JSON（可送到 Jaeger / Zipkin 等後端）：")
print(json.dumps(exported, ensure_ascii=False, indent=2))

---
## Part 5：告警與預算控制

### 5.1 AlertManager 設計

告警系統的三個要素：
1. **Rule**：條件（e.g., p95_latency > 3000ms）
2. **Severity**：嚴重程度（WARNING / CRITICAL）
3. **Action**：觸發後做什麼（印出、傳 Slack、熔斷）

In [ ]:
@dataclass
class AlertRule:
    name: str
    condition: Any          # callable(collector) -> bool
    severity: str           # WARNING | CRITICAL
    message_template: str   # f-string，可引用 {value}


@dataclass
class Alert:
    fired_at: str
    rule_name: str
    severity: str
    message: str


class AlertManager:
    """告警管理器：注冊規則，每次請求後自動評估"""

    def __init__(self):
        self._rules: List[AlertRule] = []
        self._fired: List[Alert] = []

    def register(
        self,
        name: str,
        condition,           # callable: (MetricsCollector) -> (bool, value)
        severity: str,
        message_template: str,
    ):
        self._rules.append(AlertRule(name, condition, severity, message_template))

    def evaluate(self, collector: MetricsCollector) -> List[Alert]:
        """評估所有規則，觸發的告警會被記錄並印出"""
        fired_now = []
        for rule in self._rules:
            try:
                triggered, value = rule.condition(collector)
            except Exception:
                continue
            if triggered:
                msg = rule.message_template.format(value=value)
                alert = Alert(
                    fired_at=datetime.datetime.utcnow().isoformat() + "Z",
                    rule_name=rule.name,
                    severity=rule.severity,
                    message=msg,
                )
                self._fired.append(alert)
                fired_now.append(alert)
                icon = "🔴" if rule.severity == "CRITICAL" else "🟡"
                print(f"{icon} [{rule.severity}] {rule.name}: {msg}")
        return fired_now

    def all_alerts(self) -> List[Alert]:
        return list(self._fired)


# ─── 定義告警規則 ──────────────────────────────
alert_manager = AlertManager()

# 規則 1：P95 延遲超過 3000ms → WARNING
alert_manager.register(
    name="high_p95_latency",
    condition=lambda c: (c.percentile(95) > 3000, round(c.percentile(95), 0)),
    severity="WARNING",
    message_template="P95 延遲 {value}ms 超過閾值 3000ms",
)

# 規則 2：每日推算成本超過 $10 → CRITICAL
alert_manager.register(
    name="daily_cost_exceeded",
    condition=lambda c: (
        c.summary().get("projected_daily_usd", 0) > 10.0,
        round(c.summary().get("projected_daily_usd", 0), 2),
    ),
    severity="CRITICAL",
    message_template="推算每日成本 ${value} 超過預算 $10",
)

# 規則 3：平均 tokens/sec < 5（模型太慢）→ WARNING
alert_manager.register(
    name="low_throughput",
    condition=lambda c: (
        c.summary().get("avg_tokens_per_sec", 999) < 5,
        round(c.summary().get("avg_tokens_per_sec", 0), 1),
    ),
    severity="WARNING",
    message_template="平均 tokens/sec={value} 低於閾值 5",
)

print("✅ AlertManager 初始化完成，已注冊", len(alert_manager._rules), "條規則")

### 5.2 Cost Circuit Breaker（成本斷路器）

In [ ]:
class CostCircuitBreaker:
    """
    成本斷路器：當日累積成本超過預算時，阻止後續 LLM 呼叫。
    使用方式：在每次 LLM 呼叫前呼叫 check()。
    """

    def __init__(self, daily_budget_usd: float = 10.0):
        self.daily_budget_usd = daily_budget_usd
        self._daily_spend: float = 0.0
        self._tripped: bool = False
        self._trip_reason: Optional[str] = None

    def record_spend(self, cost_usd: float):
        self._daily_spend += cost_usd
        if self._daily_spend >= self.daily_budget_usd:
            self._tripped = True
            self._trip_reason = (
                f"累積成本 ${self._daily_spend:.4f} "
                f"已超過每日預算 ${self.daily_budget_usd}"
            )

    def check(self):
        """若斷路器已跳脫，拋出例外"""
        if self._tripped:
            raise RuntimeError(f"🔴 Circuit Breaker OPEN: {self._trip_reason}")

    @property
    def is_open(self) -> bool:
        return self._tripped

    @property
    def daily_spend(self) -> float:
        return self._daily_spend

    def reset(self):
        """每日 00:00 重置（模擬）"""
        self._daily_spend = 0.0
        self._tripped = False
        self._trip_reason = None
        print("✅ Circuit Breaker 已重置")


breaker = CostCircuitBreaker(daily_budget_usd=10.0)

# ─── 模擬超過預算 ─────────────────────────────
print("模擬花費 $9.99...")
breaker.record_spend(9.99)
print(f"  is_open={breaker.is_open}  spend=${breaker.daily_spend}")

print("再花費 $0.02...")
breaker.record_spend(0.02)
print(f"  is_open={breaker.is_open}  spend=${breaker.daily_spend}")

# 嘗試呼叫
try:
    breaker.check()
    print("LLM 呼叫允許")
except RuntimeError as e:
    print(str(e))

breaker.reset()

---
## Part 6：整合示範

整合所有組件，對 20 個模擬查詢跑完整的可觀測性 Pipeline：

```
Query
  └─ CostCircuitBreaker.check()
  └─ rag_with_tracing()  ← Span Tree
       ├─ PipelineLogger（結構化日誌）
       └─ MetricsCollector（指標）
  └─ AlertManager.evaluate()
  └─ CostCircuitBreaker.record_spend()
```

In [ ]:
# ─── 全整合 RAG 函式 ───────────────────────────

def full_observability_rag(
    query: str,
    pipeline_logger: PipelineLogger,
    collector: MetricsCollector,
    tracer_inst: Tracer,
    alert_mgr: AlertManager,
    cb: CostCircuitBreaker,
    top_k: int = 3,
) -> Optional[Dict]:
    """帶有完整可觀測性的 RAG pipeline"""

    # 斷路器檢查
    try:
        cb.check()
    except RuntimeError as e:
        print(f"  ⛔ 已被 Circuit Breaker 阻擋: {e}")
        return None

    trace_id = uuid.uuid4().hex[:16]
    root = tracer_inst.start_span("rag_pipeline", trace_id=trace_id, query=query[:40])

    # Embedding
    emb_span = tracer_inst.start_span("embedding", trace_id=trace_id, parent_span=root)
    q_emb = embed([query])[0]
    emb_span.end(model=EMBED_MODEL)
    pipeline_logger.log(trace_id[:8], "embedding", emb_span.duration_ms)

    # Retrieval
    ret_span = tracer_inst.start_span("retrieval", trace_id=trace_id, parent_span=root)
    chroma_res = collection.query(query_embeddings=[q_emb], n_results=top_k)
    docs = chroma_res["documents"][0]
    ret_span.end(top_k=top_k, returned=len(docs))
    pipeline_logger.log(trace_id[:8], "retrieval", ret_span.duration_ms, docs_returned=len(docs))

    # Rerank（模擬）
    rerank_span = tracer_inst.start_span("rerank", trace_id=trace_id, parent_span=root)
    time.sleep(random.uniform(0.005, 0.02))
    rerank_span.end(strategy="mock")
    pipeline_logger.log(trace_id[:8], "rerank", rerank_span.duration_ms)

    # LLM
    context = "\n".join(f"- {d}" for d in docs)
    messages = [
        {"role": "system", "content": "你是可觀測性專家，請用繁體中文簡短回答。"},
        {"role": "user",   "content": f"知識庫：\n{context}\n\n問題：{query}"},
    ]
    llm_span = tracer_inst.start_span("llm_generation", trace_id=trace_id, parent_span=root)
    resp = client.chat.completions.create(
        model=CHAT_MODEL, messages=messages, max_tokens=150, temperature=0.3
    )
    usage = resp.usage
    llm_span.end(tokens_in=usage.prompt_tokens, tokens_out=usage.completion_tokens)
    pipeline_logger.log(
        trace_id[:8], "llm_generation", llm_span.duration_ms,
        tokens_in=usage.prompt_tokens, tokens_out=usage.completion_tokens
    )

    root.end()

    # 指標
    m = collector.record(
        request_id=trace_id[:8],
        tokens_in=usage.prompt_tokens,
        tokens_out=usage.completion_tokens,
        latency_ms=root.duration_ms,
        llm_latency_ms=llm_span.duration_ms,
    )

    # 告警評估
    alert_mgr.evaluate(collector)

    # 更新斷路器
    cb.record_spend(m.cost_usd)

    return {
        "trace_id": trace_id,
        "answer": resp.choices[0].message.content,
        "metric": m,
    }


print("✅ full_observability_rag() 定義完成")

In [ ]:
# ─── 執行 20 個模擬查詢 ──────────────────────

QUERIES_20 = [
    "什麼是可觀測性？",
    "Trace 和 Log 有什麼差別？",
    "如何計算 tokens per second？",
    "P95 延遲是什麼意思？",
    "為什麼需要結構化日誌？",
    "Span 包含哪些欄位？",
    "如何設定告警規則？",
    "Circuit Breaker 的用途是什麼？",
    "如何推算每日 LLM 成本？",
    "什麼是 trace_id？",
    "RAG 系統延遲來源有哪些？",
    "如何追蹤快取命中率？",
    "OpenTelemetry 是什麼？",
    "如何避免告警疲勞？",
    "parent_span_id 的作用是什麼？",
    "如何衡量 LLM 的 throughput？",
    "cost-per-request 如何計算？",
    "分散式追蹤的三個核心概念是什麼？",
    "Metrics 和 Traces 有什麼互補關係？",
    "如何用 span 找出 RAG 的效能瓶頸？",
]

full_logger   = PipelineLogger()
full_metrics  = MetricsCollector()
full_tracer   = Tracer()
full_alerts   = AlertManager()
full_breaker  = CostCircuitBreaker(daily_budget_usd=10.0)

# 注冊告警規則
full_alerts.register(
    "high_p95_latency",
    lambda c: (c.percentile(95) > 3000, round(c.percentile(95), 0)),
    "WARNING",
    "P95={value}ms 超過 3000ms",
)
full_alerts.register(
    "daily_budget_exceeded",
    lambda c: (
        c.summary().get("projected_daily_usd", 0) > 10.0,
        round(c.summary().get("projected_daily_usd", 0), 2),
    ),
    "CRITICAL",
    "推算每日成本=${value} 超過 $10",
)

results_20 = []
for i, q in enumerate(QUERIES_20, 1):
    print(f"\n[{i:02d}/{len(QUERIES_20)}] {q}")
    r = full_observability_rag(
        q, full_logger, full_metrics, full_tracer, full_alerts, full_breaker
    )
    if r:
        m = r["metric"]
        print(f"     → latency={m.latency_ms:.0f}ms  cost=${m.cost_usd:.6f}")
        results_20.append(r)

print(f"\n✅ 完成 {len(results_20)}/{len(QUERIES_20)} 個查詢")

In [ ]:
# ─── 儀表板：指標摘要表 ───────────────────────
s = full_metrics.summary()

print("\n" + "═" * 55)
print("  📊  可觀測性儀表板摘要")
print("═" * 55)
print(f"  {'請求總數':<28} {s['requests']}")
print(f"  {'平均延遲':<28} {s['avg_latency_ms']} ms")
print(f"  {'P50 延遲':<28} {s['p50_latency_ms']} ms")
print(f"  {'P95 延遲':<28} {s['p95_latency_ms']} ms")
print(f"  {'平均 tokens_in':<28} {s['avg_tokens_in']}")
print(f"  {'平均 tokens_out':<28} {s['avg_tokens_out']}")
print(f"  {'平均 tokens/sec':<28} {s['avg_tokens_per_sec']}")
print(f"  {'平均 cost/request':<28} ${s['avg_cost_usd']:.6f}")
print(f"  {'總花費':<28} ${s['total_cost_usd']:.6f}")
print(f"  {'推算每日花費 (1000 req)':<28} ${s['projected_daily_usd']:.4f}")
print(f"  {'推算每月花費':<28} ${s['projected_monthly_usd']:.2f}")
print("═" * 55)
print(f"  告警總計觸發次數: {len(full_alerts.all_alerts())}")
print(f"  斷路器狀態: {'OPEN 🔴' if full_breaker.is_open else 'CLOSED ✅'}")
print(f"  本日累積花費: ${full_breaker.daily_spend:.6f}")
print("═" * 55)

In [ ]:
# ─── 儀表板：延遲圖 + 成本圖 ─────────────────
plot_metrics(full_metrics)

In [ ]:
# ─── 顯示第一個請求的 Trace Tree ─────────────
if results_20:
    first_trace = results_20[0]["trace_id"]
    full_tracer.print_trace_tree(first_trace)
    print("\n📌 該請求的答案：")
    print(results_20[0]["answer"])

In [ ]:
# ─── 顯示所有觸發的告警 ──────────────────────
alerts = full_alerts.all_alerts()
if alerts:
    print(f"\n🚨 共觸發 {len(alerts)} 次告警：")
    for a in alerts:
        icon = "🔴" if a.severity == "CRITICAL" else "🟡"
        print(f"  {icon} [{a.fired_at[:19]}] [{a.severity}] {a.rule_name}: {a.message}")
else:
    print("\n✅ 未觸發任何告警")

In [ ]:
# ─── 延遲細項分解圖 ───────────────────────────
# 從 tracer 取出每個 trace 的各 span 延遲

def plot_latency_breakdown(tracer_inst: Tracer, n_traces: int = 10):
    """繪製 RAG 各階段延遲分解圖（stacked bar）"""
    # 整理每個 trace 的 spans
    from collections import defaultdict

    spans_by_trace: Dict[str, Dict[str, float]] = defaultdict(dict)
    for span in tracer_inst.all_spans():
        if span.name != "rag_pipeline":  # 只取子 spans
            spans_by_trace[span.trace_id][span.name] = span.duration_ms

    trace_ids = list(spans_by_trace.keys())[:n_traces]
    if not trace_ids:
        print("⚠️ 無 span 資料")
        return

    steps = ["embedding", "retrieval", "rerank", "llm_generation"]
    step_labels = ["Embedding", "Retrieval", "Rerank", "LLM 生成"]
    colors = ["#4C72B0", "#55A868", "#C44E52", "#DD8452"]

    data = {step: [] for step in steps}
    for tid in trace_ids:
        for step in steps:
            data[step].append(spans_by_trace[tid].get(step, 0))

    x = np.arange(len(trace_ids))
    fig, ax = plt.subplots(figsize=(14, 5))
    bottom = np.zeros(len(trace_ids))

    for step, label, color in zip(steps, step_labels, colors):
        vals = np.array(data[step])
        ax.bar(x, vals, bottom=bottom, label=label, color=color, alpha=0.85)
        bottom += vals

    ax.set_xlabel("請求序號")
    ax.set_ylabel("延遲 (ms)")
    ax.set_title("RAG Pipeline 各階段延遲分解")
    ax.set_xticks(x)
    ax.set_xticklabels([f"#{i+1}" for i in range(len(trace_ids))], rotation=45)
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


plot_latency_breakdown(full_tracer)

---
## Part 7：面試 Q&A

### Q1：在 LLM 系統中，可觀測性（Observability）與傳統 Web 服務有何不同？

**A：** 傳統 Web 服務的可觀測性關注 RPS、Error Rate 和 Latency（俗稱 RED 指標）。LLM 系統雖然也需要這些，但還有幾個獨特挑戰：

1. **成本是一等公民**：每次請求的費用不固定（取決於 token 數量），必須追蹤 cost-per-request 和每日/每月成本推算。
2. **延遲分布特殊**：LLM 的延遲高度依賴輸出長度（TTFT vs TPOT），P99 可能是 P50 的 10 倍，不能只看平均。
3. **品質指標難量化**：hallucination rate、faithfulness 等指標無法從 HTTP status code 推斷，需要額外評估層。
4. **Token 是關鍵資源**：tokens/sec（throughput）、context utilization 是 LLM 系統獨有的 SLI。

**面試技巧**：提到「三支柱（Logs/Metrics/Traces）+ LLM-specific metrics」，展示你知道不能只套用傳統方案。

### Q2：在 RAG 系統中，你會追蹤哪些關鍵指標？如何設定告警閾值？

**A：** 按優先級分類：

**可靠性（Reliability）**
- `error_rate`：pipeline 各步驟錯誤率（目標 < 1%）
- `retrieval_empty_rate`：檢索返回 0 結果的比例（表示知識庫覆蓋不足）

**效能（Performance）**
- `p50_latency` / `p95_latency`：建議以 P95 < 3s 作為 SLO
- `tokens_per_sec`：< 10 tokens/sec 可能表示推論基礎設施過載
- `ttft`（Time To First Token）：影響流式輸出的感知體驗

**成本（Cost）**
- `cost_per_request`：超過預期成本的請求（可能是 context 被填太滿）
- `projected_daily_usd`：推算每日成本，超過預算觸發 CRITICAL

**品質（Quality）**
- `faithfulness_score`：需結合離線評估（NB04 的方法）
- `cache_hit_rate`：Semantic cache 命中率低 → 快取策略需調整

**告警閾值設定原則**：先收集 1-2 週基線數據，以 baseline P95 的 1.5x 作為 WARNING、2x 作為 CRITICAL，避免一開始就設定太嚴格導致告警疲勞。

### Q3：生產環境中 RAG 突然延遲上升，你會如何 debug？

**A：** 系統化的 debug 流程，從粗到細：

**Step 1：確認問題範圍**
- 查看 metrics：是所有請求都慢，還是特定 query 類型？
- 看延遲分布：P50 沒變但 P95 飆高 → 偶發性問題（通常是外部依賴）

**Step 2：用 Trace 定位哪個 Span 耗時**
- 篩選延遲最高的 10 個 trace，比較各 span 的耗時比例
- 如果 `embedding` span 突然變慢 → 向量資料庫或 embedding API 有問題
- 如果 `llm_generation` span 突然變慢 → LLM API 服務問題或 input tokens 激增

**Step 3：關聯外部事件**
- 查看 tokens_in 是否異常增大（context stuffing 導致 LLM 慢）
- 查看 retrieval 返回文件數是否正常
- 確認是否有部署或設定變更

**Step 4：結構化日誌分析**
- 過濾 `step_name=retrieval` 的日誌，計算 duration_ms 的 P95 趨勢
- 比對問題發生前後的 token 分布

**常見根因**：向量資料庫 index 沒有熱身（cold start）、LLM 上游限流、Prompt template 改動導致 context 長度暴增。

### Q4：如何在生產環境中做 LLM 成本控制？Circuit Breaker 的優缺點是什麼？

**A：** 成本控制的多層防線：

**預防層（事前）**
- **Max tokens 限制**：在 API 呼叫設 `max_tokens`，避免意外生成超長輸出
- **Context 壓縮**：只送必要的 chunks，不要把全部 retrieved docs 全填入
- **Semantic Cache**：相似查詢直接返回快取結果，不呼叫 LLM
- **Model 分級**：簡單問題用小模型（gpt-4o-mini），複雜問題才升級

**監控層（事中）**
- 每次請求後更新 cost accumulator
- 預算用量達 80% 時發 WARNING，100% 時發 CRITICAL

**熔斷層（事後）**
- **Circuit Breaker 優點**：當預算耗盡時自動阻止後續請求，防止帳單無限增長；狀態可持久化，支援每日自動重置
- **Circuit Breaker 缺點**：
  - 一刀切：所有用戶都被阻擋，無法做優先級區分（VIP 用戶應該要能繼續）
  - 狀態管理複雜：多實例部署時需要共享狀態（Redis）
  - 誤判風險：若 cost estimator 有 bug，可能提早觸發

**更精細的做法**：使用 **Rate Limiting per user/tenant**，搭配 **Cost Quota per tier**，讓不同方案的用戶有不同的每日 token 預算，而不是全局斷路。

**面試亮點**：主動提到「多實例環境下 Circuit Breaker 需要 distributed state（Redis）」，展現對生產環境複雜性的理解。

---
## 小結

本筆記實作了 RAG 系統可觀測性的完整基礎設施：

| 組件 | 類別 | 功能 |
|------|------|------|
| 結構化日誌 | `PipelineLogger` | 每個步驟輸出 JSON log，可按欄位查詢 |
| 指標收集 | `MetricsCollector` | 追蹤 latency / cost / tokens/sec，計算 P50/P95 |
| 分散式追蹤 | `Tracer` / `Span` | 建立 Span Tree，視覺化各階段耗時 |
| 告警管理 | `AlertManager` | 可注冊任意規則，觸發 WARNING/CRITICAL |
| 成本斷路器 | `CostCircuitBreaker` | 預算耗盡時自動阻擋 LLM 呼叫 |

**下一步建議：**
- 將 log/trace 接入 Elasticsearch + Kibana 做視覺化
- 使用真實的 OpenTelemetry SDK 替換自製 Tracer，送至 Jaeger
- 加入 Prometheus metrics endpoint，配合 Grafana 儀表板
- 結合 NB04 的評估框架，追蹤 faithfulness 和 answer relevance 趨勢